## Context Aware & Stateful Tools

### We'll use a E-commerce Customer Support AI Agent example to understand need of ToolRuntime & How to Use It.

**Scenario:** : A customer asks:
 >   "What's the status of my order 12345?"

The agent has multiple tools:
* get_order_details
* cancel_order
* send_email

We'll use this as example to understand ToolRuntime concept.

#### Before goint through example usecases, it's worth noting some information arounf Agent s and Langchain.
**Langchain Provides 2 Reserved arguments for Tools**
| Parameter name | Purpose |
|----------|---------------|
| config  | Reserved for passing RunnableConfig to tools internally |
| runtime: ToolRuntime | Reserved for ToolRuntime parameter (accessing state, context, store) |


#### Information in `ToolRuntime` will NOT BE VISIBLE TO LLMs**
* So if we have some secret information or user_ids etc, we can use ToolRuntime to hide that information.
* Prove the hiding claim in below code block --> `runtime` will NOT appear here, only `action` will be visible


In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def get_last_order_details(action: str, runtime: ToolRuntime) -> str:
    """Find the last order details of the customer in this conversation."""
    return "No orders found."

print("Tool schema seen by the model:", get_last_order_details.args)

Tool schema seen by the model: {'customer_id': {'title': 'Customer Id', 'type': 'string'}}


### ToolRuntime: provides below Runtime information
```python
    def get_orders(runtime: ToolRuntime):
        runtime.state  # Short-term memory - mutable data: Access conversation history, track tool call counts
        runtime.store  # Long-term memory - Save user preferences, maintain knowledge base
        runtime.context # Personalize responses based on user identity e.g runtime.context.user_id
        runtime.config # Access callbacks, tags, and metadata
        runtime.stream_writer # Emit real-time updates during tool execution
        runtime.execution_info # Process and retry information for the current execution (thread ID, run ID, attempt number)
        runtime.tool_call_id # 
```

### In Langchain there are 2 Ways LLMs see the exiting tools
* `model.bind_tools([list_of_tools])` This only makes the model **AWARE** which tools exist — it **Does Not RUN** anything.
* Langchain's Agent **`create_agent`** this actually runs the tool and looping back for a final answer.

<img src="../../assets/bind_vs_create_agent.png" width="1200" height="150">


In [59]:
import os
import json
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, BaseMessage
from typing import TypedDict, Sequence,Annotated
from pydantic import BaseModel
import operator

load_dotenv()

True

## 1. Access State - Mutable Short Term Memory: `runtime.state`

In [52]:

# 1. Define the Custom Agent State structure
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    user_membership_tier: str  # <--- Our custom state field

# 2. Define the tool using ToolRuntime
@tool()
def apply_user_discount(item_id: str, runtime: ToolRuntime) -> str:
    """Calculate an item's price by looking up the user's active membership tier."""
    # Pull the custom state parameter out of the unified runtime object
    current_state = runtime.state
    user_tier = current_state.get("user_membership_tier", "guest")
    
    base_price = 100.0
    discount = 0.20 if user_tier == "premium" else 0.0
    final_price = base_price * (1 - discount)
    
    return f"Item {item_id} final price for {user_tier} tier: ${final_price:.2f}"

# 4. Create the Agent using create_agent
stateful_agent = create_agent(
    model='openai:gpt-5-nano',
    tools=[apply_user_discount],
    state_schema=AgentState
)

In [53]:

premium_input = {
        "messages": [("user", "How much does item 'xyz-789' cost for me?")],
        "user_membership_tier": "premium"
    }
normal_input = {
        "messages": [("user", "How much does item 'xyz-789' cost for me?")],
        "user_membership_tier": "guest"
    }
result_premium_user = stateful_agent.invoke(premium_input)
result_normal_user = stateful_agent.invoke(normal_input)

In [58]:
result_premium_user

{'messages': [('user', "How much does item 'xyz-789' cost for me?"),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 283, 'prompt_tokens': 146, 'total_tokens': 429, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E8IM83ajI56NFhgb6GQncQ7TI5mu7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fc0ca-002e-7000-b6f8-ffe3149c43aa-0', tool_calls=[{'name': 'apply_user_discount', 'args': {'item_id': 'xyz-789'}, 'id': 'call_ZpsteFgO7MVHHzndFCjFXXV0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 146, 'output_tokens': 283, 'total_tokens': 429, 'input_token_det

In [55]:
result_normal_user

{'messages': [('user', "How much does item 'xyz-789' cost for me?"),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 219, 'prompt_tokens': 146, 'total_tokens': 365, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E8IMDh2t1X1lN5xwKUzwUuoWA4BFn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fc0ca-1554-7892-ad6d-14867a58d56f-0', tool_calls=[{'name': 'apply_user_discount', 'args': {'item_id': 'xyz-789'}, 'id': 'call_NLzYIYauQY1ZvwSEX9utiuzl', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 146, 'output_tokens': 219, 'total_tokens': 365, 'input_token_det

## 2. Context - Static Run Configuration `runtime.context`

In [63]:
ORDERS_DATABASE = {
    "user_123": {
       "ORD001":
       { 
            "name": "Shivam",
            "account_type": "Premium",
            "order_status": 'Delivered',
            "total_amount": 5000
        },
        "ORD002":
        { 
            "name": "Shivam",
            "account_type": "Premium",
            "order_status": 'Placed',
            "total_amount": 1000,
            "estimated_delivery": "2024-06-15"
        }
    },
    
    "user_456": {
       "ORD005":
       {
            "name": "Satyam",
            "account_type": "Standard",
            "order_status": 'Placed',
            "total_amount": 1200,
            "estimated_delivery": "2025-04-15"
       }
    }
}

class UserContext(BaseModel):
    user_id: str

@tool
def get_order_status(runtime: ToolRuntime[UserContext], order_id: str) -> str:
    """Retrieve the order status for the user based on their user_id from the runtime context.
    Args:
        order_id (str): The ID of the order to retrieve the status for.
    """
    user_id = runtime.context.user_id
    if user_id in ORDERS_DATABASE and order_id in ORDERS_DATABASE[user_id]:
        order = ORDERS_DATABASE[user_id][order_id]
        order_info = f"Order ID: {order_id}\nName: {order['name']}\nType: {order['account_type']}\nOrder Status: {order['order_status']}\nTotal Amount: ${order['total_amount']}"
        if 'estimated_delivery' in order:
            order_info += f"\nEstimated Delivery: {order['estimated_delivery']}"
        return order_info
    return "User or Order not found"

contextual_agent = create_agent(
    model='openai:gpt-5-nano',
    tools=[get_order_status],
    context_schema=UserContext,
    system_prompt="You are a helpful assistant that provides order status information based on the user's context."
)


In [64]:
result = contextual_agent.invoke(
    {"messages": [{"role": "user", "content": "I have an order with order id ORD002. Can you tell me its status? and what is the estimated delivery date?"}]},
    context=UserContext(user_id="user_123")
)

In [65]:
result

{'messages': [HumanMessage(content='I have an order with order id ORD002. Can you tell me its status? and what is the estimated delivery date?', additional_kwargs={}, response_metadata={}, id='3996d684-faa1-42cb-8c3a-4f23098f4045'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 154, 'prompt_tokens': 199, 'total_tokens': 353, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E8Ire01BKfziBjqYG5EV1bYbLaU15', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fc0e7-ced4-7a12-9508-3bc67bd9efb1-0', tool_calls=[{'name': 'get_order_status', 'args': {'order_id': 'ORD002'}, 'id': 'call_5vY8bZ91

```text
    User:  
    I have an order with order id ORD002. Can you tell me its status? And what is the estimated delivery date.

    Assistant:  
    (get_order_status tool was called with: order_id = ORD002)

    Tool (get_order_status):  
    Order ID: ORD002
    Name: Shivam
    Type: Premium
    Order Status: Placed
    Total Amount: $1000
    Estimated Delivery: 2024‑06‑15

    Assistant:  
    Here are the details for ORD002:

    Status: Placed

    Estimated Delivery: 2024‑06‑15

    Name: Shivam

    Type: Premium

    Total Amount: $1000

    Would you like me to set up delivery notifications or check for any updates on this order.
```

## 3. Access Sore - Long-term memory: `runtime.store`
* Accessing / Remembering a Customer's Cart information Across Entirely Separate Visits using `runtime.store`
* `runtime.state` only covers Current conversation. 
* For memory that survives across completely separate sessions, a `Store` is attached to the agent and reached via `runtime.store`.
* For production deployments, use persistent store implementation like `PostgresStore` instead of `InMemoryStore`

In [ ]:
from langgraph.store.memory import InMemoryStore

cart_store = InMemoryStore()

@tool
def store_item_to_cart(customer: str, item: str, runtime: ToolRuntime) -> str:
    """Save an item to a customer's cart for future visits."""
    # Fetch existing items
    result = runtime.store.get((customer, "preferences"), "items")
    items = result.value["value"] if result else []

    # Append new item
    items.append(item)

    # Store updated list
    runtime.store.put((customer, "preferences"), "items", {"value": items})
    return f"Got it -- I've added {item} to your cart."

@tool
def get_items_from_cart(customer: str, runtime: ToolRuntime) -> str:
    """Recall the items in a customer's cart, if we've saved them before."""
    result = runtime.store.get((customer, "preferences"), "items")
    return result.value["value"] if result else "We don't have any items saved for this customer yet."

memory_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[store_item_to_cart, get_items_from_cart],
    store=cart_store,
)

In [78]:
memory_agent.invoke({"messages": [("user", "Hi, I'm customer shivam, I want to add an item Macbook to my cart.")]})
result = memory_agent.invoke({"messages": [("user", "What items do I have in my cart? I'm shivam.")]})

In [79]:
print(result["messages"][-1].content)

Hi Shivam! You currently have the following item in your cart: Macbook.

Would you like to add more items, remove something, or proceed to checkout?


In [80]:
result

{'messages': [HumanMessage(content="What items do I have in my cart? I'm shivam.", additional_kwargs={}, response_metadata={}, id='07ff7b42-ad65-4af1-a400-99a0f0bb567c'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 218, 'prompt_tokens': 179, 'total_tokens': 397, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E8JKdh7OKHwm8DEBiJiJDuMfnrozs', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fc103-3df0-7d60-9e7c-d68039894834-0', tool_calls=[{'name': 'get_items_from_cart', 'args': {'customer': 'shivam'}, 'id': 'call_Pe04Tco81EBZAcgLLvQo4aa2', 'type': 'tool_call'}], invalid_tool_call

In [81]:

memory_agent.invoke({"messages": [("user", "Hi, I'm customer shivam, can you add cricket bat in my cart.")]})
result = memory_agent.invoke({"messages": [("user", "What items do I have in my cart? I'm shivam.")]})

In [82]:
print(result["messages"][-1].content)

Here are the items in your cart, Shivam:
- Macbook
- cricket bat

Would you like to remove anything, view details, or proceed to checkout? I can also add more items if you’d like.


#### Inspecting the Store Directly: `.search()`
* Beyond `.get()` (fetch one specific key) and `.put()` (save one)
* `Store` also supports `.search()`that lists every item saved under a given namespace, without needing to know each exact key in advance. 
* Genuinely useful for debugging or building an admin view of what's been saved.

In [ ]:
items = cart_store.search(("shivam", "preferences"))
for item in items:
    print(item)

Item(namespace=['shivam', 'preferences'], key='items', value={'value': ['Macbook', 'cricket bat']}, created_at='2026-08-02T05:47:45.711344+00:00', updated_at='2026-08-02T05:47:45.711347+00:00', score=None)


## 4. Two Stream Writer : `execution_info`
`execution_info` gives the current thread/run/retry identity. 
```python
    @tool
    def log_execution_context(runtime: ToolRuntime) -> str:
        """Log execution identity information."""
        info = runtime.execution_info
        print(f"Thread: {info.thread_id}, Run: {info.run_id}")
        print(f"Attempt: {info.node_attempt}")
        return "done"
```

### Tool Return Values
1. **Return a string**
    * Return a string when the tool should provide plain text for the model and use in its next response.
    * The return value is converted to a ToolMessage.
    * The model sees that text and decides what to do next.

2. **Return an object**
    * Return an object (for example, a dict) when your tool produces structured data for model to inspect.
    * The object is serialized and sent back as tool output.
    * The model can read specific fields and reason over them.

3. **Return multimodal content**

    When the model supports multimodal tool results, the tool can return content blocks so the model receives text, images, and other media in one tool result
    ```python
        @tool
        def capture_screenshot() -> list[dict]:
            """Capture a screenshot of the current page."""
            return [
                {"type": "text", "text": "Screenshot of the current page:"},
                {"type": "image", "url": "https://example.com/page.png"},
            ]
    ```
    * The return value is converted to a ToolMessage with multimodal content.
    * Use `message.content_blocks` to read the block list after the tool runs.
    * Check model’s capabilities before returning images, audio, or video.

4. **Return directly from a tool `return_direct`**
    * Sometimes a tool's raw output IS the final answer — no rephrasing needed.
    * Use when the exact tool wording matters that must never be paraphrased.
    * Declare `@tool(return_direct=True)`, the agent returns the tool output directly without another LLM call. 
    * With this Agent's observation step will be skipped and Once this tool is called, response will directly be sent to user.
    * In this case Last message will be a `ToolMessage` instead of `AIMessage`

In [84]:
@tool(return_direct=True)
def get_exact_refund_policy() -> str:
    """Tell the refund policy."""
    return "Refunds are applicable till before the delivery date. No refunds after that."

direct_agent = create_agent(model="openai:gpt-5-nano", tools=[get_exact_refund_policy])
result = direct_agent.invoke({"messages": [("user", "What's your refund policy?")]})
result

{'messages': [HumanMessage(content="What's your refund policy?", additional_kwargs={}, response_metadata={}, id='058e471e-1a87-47cb-8f23-c7816aec79b6'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 150, 'prompt_tokens': 125, 'total_tokens': 275, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E8Jm6xGc1xoLOKN3GiIBnsq0fj1wd', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fc11d-37f5-7e91-8469-34a63b57dafc-0', tool_calls=[{'name': 'get_exact_refund_policy', 'args': {}, 'id': 'call_47iqTJgWx7zRzWIGctT0OSJW', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_token

## Dynamic tool selection
Dynamic tools available to the agent, is modified at runtime rather than defined all upfront.
With dynamic tools, the set of tools available to the agent is modified at runtime rather than defined all upfront.

<img src="../../assets/many_tools.png" width="400" height="250">

* Suppose we have 300 tools and each tool has information of ~500 tokens
* Showing all available tools to LLM will send unnecessary tool tokens, in this case 300 * 500 = 1,50,000 Total tokens for tools information only will be sent to LLM in each user query.
* Another problem is our Model will also get confused, too many tools may overwhelm the model (overload context) and increase errors
* Dynamic tool selection enables adapting the available toolset based on 
    * authentication state
    * user permissions etc.

###  Analogy: Access to premium Entertainment services
E-commerce have some services (Amazon Prime) which are applicable to only Premium users. e.g. Entertainment Services.
* A naive fix is telling the model in the system prompt "don't offer this to Non-Premium users" 
* But this relies on the model choosing to follow an instruction, which might not strictly follow.

**The Fix: `wrap_model_call` — Making Entertainment Tool Genuinely Disappear For Non-Premium users**

## Chapter 2 Summary

- A tool is a phone line out of an otherwise stuck model; the docstring is its entire pitch for
  when to use it.
- Check for a **prebuilt tool** (like Tavily search) before writing one from scratch.
- `args_schema` handles complex inputs; `config` and `runtime` are reserved argument names.
- `.bind_tools()` only makes a model aware of tools — `create_agent` actually runs them.
- `ToolRuntime` gives hidden access to `state`, `context`, `store` (including `.search()` to
  list saved items), `execution_info`, and `server_info` — invisible to the model.
- `return_direct=True` skips the model's final rephrasing pass.
- Tools can be filtered out of a model's awareness entirely, based on runtime conditions — a
  genuinely stronger guarantee than instructing the model not to use something.
